In [57]:
import pandas as pd

df_train = pd.read_csv('../data/raw/train.csv')
df_val = pd.read_csv('../data/raw/val.csv')
df_test = pd.read_csv('../data/raw/test.csv')

In [58]:
import re
import emoji

SKIN_TONE = re.compile(r"_(?:light|medium-light|medium|medium-dark|dark)_skin_tone")

def clean_text(text, keep_hashtags=True, keep_emoji=True):
    text = re.sub(r"https?://\S+|www\.\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#" if keep_hashtags else r"#\w+", "", text)
    text = re.sub(r"<[^>]+>", "", text)
    if keep_emoji:
        text = emoji.demojize(text, delimiters=(" ", " "))
        text = SKIN_TONE.sub("", text)
        text = text.replace("_", " ")
    else:
        text = emoji.replace_emoji(text, replace=" ")
    text = re.sub(r"[^A-Za-z'\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [59]:
import re 
import contractions
import unicodedata
from nltk.tokenize import TweetTokenizer

tweet_tokenizer  = TweetTokenizer()

def normalize_text(text):
    text = unicodedata.normalize("NFC", text)               
    text = text.replace("\r\n", "\n") 
    text = text.casefold()
    text = re.sub(r"(.)\1{2,}", r"\1\1", text) 
    text = text.split()
    text = [contractions.fix(t) for t in text]
    text = " ".join([t for t in text])
    tokens = tweet_tokenizer.tokenize(text)
    return tokens

In [60]:
from nltk.corpus import stopwords

STOP_WORDS = set(stopwords.words("english")) - {"not", "no", "nor", "never", "up", "down"}

def remove_stopwords(tokens):
    return [t for t in tokens if t.isalpha() and t not in STOP_WORDS and len(t) > 1]

In [61]:
from nltk import WordNetLemmatizer
from nltk import pos_tag
from nltk.corpus import wordnet

lemmatizer = WordNetLemmatizer()

def to_wordnet_pos(tag):
    if tag.startswith("J"):
        return wordnet.ADJ
    if tag.startswith("V"):
        return wordnet.VERB
    if tag.startswith("R"):
        return wordnet.ADV
    return wordnet.NOUN

def lemmatize_tokens(tokens):
    return [lemmatizer.lemmatize(word, to_wordnet_pos(tag)) for word, tag in pos_tag(tokens)]

In [62]:
def text_preprocessing(text, stopwords=True, lemmatize=True, hashtags=True, emoji=True):
    text = clean_text(text, keep_hashtags=hashtags, keep_emoji=emoji)
    tokens = normalize_text(text)
    if stopwords: 
        tokens = remove_stopwords(tokens)
    if lemmatize:
        tokens = lemmatize_tokens(tokens)   
    return " ".join(tokens)

In [63]:
text = df_train['text'].sample(10)
text

1796    @user @user @user no! Kinda offended that you ...
3063    The focal points of war lie in #terrorism and ...
340     If you follow #Trump, a certified #bully there...
2724    This uber driver TRIED it with me &amp; he'll ...
2864    Mou is too jovial and trying to impress those ...
691     #aliens #zombie #gore #slash #ghost #sith #hor...
2134      Actually fuming I have nothing to wear Saturday
2807    Gonna be a loooooong year as a Browns fan. Lon...
2209    @user how do u guys determine teams? Cause I'm...
2282                         @user @user fingers of fury!
Name: text, dtype: str

In [64]:
text = df_train['text'][1304]
print(text_preprocessing(text))

happens vega not origin prevelant sadly


In [65]:
df_train['cleaned_text'] = df_train['text'].apply(text_preprocessing)
df_val['cleaned_text'] = df_val['text'].apply(text_preprocessing)
df_test['cleaned_text'] = df_test['text'].apply(text_preprocessing)

In [66]:
import os

output_path = "../data/processed"
os.makedirs(output_path, exist_ok=True)

df_train.to_csv(rf"{output_path}/train.csv", index=False)
df_val.to_csv(rf"{output_path}/val.csv", index=False)
df_test.to_csv(rf"{output_path}/test.csv", index=False)

## Testing text processing options (ablation)

In [67]:
import itertools
from sklearn.metrics import accuracy_score, f1_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

y_train, y_val = df_train["label"].values, df_val["label"].values

In [68]:
options = {
    "stopwords":     [True, False],
    "lemmatize":     [True, False],
    "hashtags":      [True, False],
    "emoji":         [True, False],
}

In [69]:
rows = []
for sw, lem, ht, em in itertools.product(*options.values()):
    prep = lambda t: text_preprocessing(t, stopwords=sw, lemmatize=lem,
                                        hashtags=ht, emoji=em)
    tr = df_train["text"].apply(prep).fillna("")
    va = df_val["text"].apply(prep).fillna("")

    vec = TfidfVectorizer(min_df=2, max_features=2000, norm="l2")
    Xtr = vec.fit_transform(tr)
    Xva = vec.transform(va)

    clf = LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42).fit(Xtr, y_train)
    pred = clf.predict(Xva)

    rows.append({"stopwords": sw, "lemmatize": lem, "hashtags": ht, "emoji": em,
                 "vocab": len(vec.vocabulary_),
                 "accuracy": accuracy_score(y_val, pred),
                 "macro_f1": f1_score(y_val, pred, average="macro")})

ablation = pd.DataFrame(rows).sort_values("macro_f1", ascending=False)
ablation.round(3)

,stopwords,lemmatize,hashtags,emoji,vocab,accuracy,macro_f1
4,True,False,True,True,2000,0.668,0.616
0,True,True,True,True,2000,0.674,0.611
8,False,True,True,True,2000,0.663,0.609
12,False,False,True,True,2000,0.658,0.606
9,False,True,True,False,2000,0.658,0.604
1,True,True,True,False,2000,0.663,0.601
5,True,False,True,False,2000,0.650,0.592
13,False,False,True,False,2000,0.626,0.570
6,True,False,False,True,2000,0.591,0.544
2,True,True,False,True,2000,0.586,0.540
